In [1]:
from google import colab
colab.drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
!unzip "/content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset.zip" -d "/content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image3055.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image3056.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image3057.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image3059.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image306.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image3060.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image3061.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Unknown/image3063.jpg  
  inflating: /content/drive/MyDrive/Colab Notebooks/Pr

In [24]:
dataset_path = "/content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset"

In [25]:
from PIL import Image
import os

deleted = 0

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        path = os.path.join(root, file)

        try:
            img = Image.open(path)
            img.verify()
        except:
            os.remove(path)
            deleted += 1
            print("Deleted:", path)

print("Total deleted:", deleted)

Total deleted: 0


In [26]:
from PIL import Image
import os

deleted = 0

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        path = os.path.join(root, file)

        try:
            img = Image.open(path)
            img.load()

        except Exception as e:
            print("Deleting:", path)

            try:
                os.remove(path)
                deleted += 1
            except:
                pass

print("Total deleted:", deleted)

Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize healthy/healthy189_.jpg
Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize healthy/healthy87_.jpg
Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize leaf beetle/leaf beetle206_.jpg
Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize leaf beetle/leaf beetle457_.jpg
Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize leaf beetle/leaf beetle572_.jpg
Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize leaf beetle/leaf beetle68_.jpg
Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize leaf beetle/leaf beetle797_.jpg
Deleting: /content/drive/MyDrive/Colab Notebooks/Projects/Final Year Project/Dataset/Maize/Maize leaf blight/leaf b

In [27]:
import os

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):

        total = 0

        for root, dirs, files in os.walk(folder_path):
            total += len(files)

        print(folder, ":", total)

Cashew : 6549
Cassava : 7508
Maize : 5289
Tomato : 5780
Unknown : 6402


In [7]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2B1
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

import numpy as np
import os

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 5
EPOCHS_FINE = 10
SEED = 42
AUTO_RESUME = True
RESUME_PREFERENCE = "last"
CONFIDENCE_THRESHOLDS = [0.70, 0.75, 0.80, 0.85, 0.90]
MARGIN_THRESHOLD = 0.10

In [ ]:
ARTIFACT_ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
RUN_NAME = "crop_classifier_efficientnetv2b1"
RUN_DIR = os.path.join(ARTIFACT_ROOT, RUN_NAME)

BEST_DIR = os.path.join(RUN_DIR, "best")
CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
RECOVERY_DIR = os.path.join(RUN_DIR, "recovery")
FINAL_DIR = os.path.join(RUN_DIR, "final")
CSV_LOG_DIR = os.path.join(RUN_DIR, "logs", "csv")
TENSORBOARD_LOG_DIR = os.path.join(RUN_DIR, "logs", "tensorboard")
BACKUP_ROOT_DIR = os.path.join(RUN_DIR, "backup")

BEST_MODEL_PATH = os.path.join(BEST_DIR, "best_model.keras")
LAST_MODEL_PATH = os.path.join(RECOVERY_DIR, "last_model.keras")
FINAL_MODEL_PATH = os.path.join(FINAL_DIR, "final_model.keras")
EPOCH_CHECKPOINT_PATTERN = os.path.join(CHECKPOINT_DIR, "model_epoch_{epoch:02d}.keras")

for path in [
    RUN_DIR,
    BEST_DIR,
    CHECKPOINT_DIR,
    RECOVERY_DIR,
    FINAL_DIR,
    CSV_LOG_DIR,
    TENSORBOARD_LOG_DIR,
    BACKUP_ROOT_DIR,
]:
    os.makedirs(path, exist_ok=True)

print(f"Artifacts directory: {RUN_DIR}")
print(f"Best model path: {BEST_MODEL_PATH}")
print(f"Recovery model path: {LAST_MODEL_PATH}")

In [ ]:
train_datagen = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    brightness_range=[0.6, 1.4],
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

class_names = list(train_generator.class_indices.keys())
print(train_generator.class_indices)

In [ ]:
base_model = EfficientNetV2B1(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

model = keras.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.2),

    layers.Dense(256, activation='relu'),

    layers.Dropout(0.2),

    layers.Dense(
        train_generator.num_classes,
        activation='softmax'
    )
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ]
)


In [ ]:
def get_stage_csv_path(stage_name):
    return os.path.join(CSV_LOG_DIR, f"{stage_name}.csv")


def get_completed_epochs(stage_name):
    csv_path = get_stage_csv_path(stage_name)
    if not os.path.exists(csv_path):
        return 0

    with open(csv_path, "r", encoding="utf-8") as f:
        completed = max(sum(1 for _ in f) - 1, 0)

    return completed


def save_recovery_snapshot(model, tag=None):
    model.save(LAST_MODEL_PATH)
    model.save(FINAL_MODEL_PATH)

    if tag:
        tagged_path = os.path.join(RECOVERY_DIR, f"{tag}.keras")
        model.save(tagged_path)
        print(f"Saved tagged recovery model: {tagged_path}")

    print(f"Saved latest recovery model: {LAST_MODEL_PATH}")
    print(f"Saved current final model: {FINAL_MODEL_PATH}")


def build_callbacks(stage_name):
    stage_backup_dir = os.path.join(BACKUP_ROOT_DIR, stage_name)
    stage_tensorboard_dir = os.path.join(TENSORBOARD_LOG_DIR, stage_name)
    stage_csv_path = get_stage_csv_path(stage_name)

    os.makedirs(stage_backup_dir, exist_ok=True)
    os.makedirs(stage_tensorboard_dir, exist_ok=True)

    return [
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=stage_backup_dir
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=BEST_MODEL_PATH,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=EPOCH_CHECKPOINT_PATTERN,
            save_best_only=False,
            save_weights_only=False,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=LAST_MODEL_PATH,
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),
        tf.keras.callbacks.CSVLogger(
            filename=stage_csv_path,
            append=os.path.exists(stage_csv_path)
        ),
        tf.keras.callbacks.TensorBoard(
            log_dir=stage_tensorboard_dir,
            histogram_freq=0,
            write_graph=False,
            update_freq="epoch",
            profile_batch=0
        ),
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
    ]

In [ ]:
def resolve_resume_model_path(prefer="last"):
    candidates = {
        "last": LAST_MODEL_PATH,
        "best": BEST_MODEL_PATH,
        "final": FINAL_MODEL_PATH,
    }

    ordered_paths = [candidates.get(prefer)] + [
        path for name, path in candidates.items() if name != prefer
    ]

    for path in ordered_paths:
        if path and os.path.exists(path):
            return path

    return None


def load_resume_model(prefer="last"):
    resume_path = resolve_resume_model_path(prefer=prefer)
    if resume_path is None:
        print("No saved model artifact found. Starting fresh.")
        return None

    print(f"Loading resume model from: {resume_path}")
    return keras.models.load_model(resume_path)

In [ ]:
if AUTO_RESUME:
    resumed_model = load_resume_model(prefer=RESUME_PREFERENCE)
    if resumed_model is not None:
        model = resumed_model
        print("Resume model loaded into current session.")
    else:
        print("No resume artifact found. Using freshly initialized model.")
else:
    print("AUTO_RESUME is disabled. Using freshly initialized model.")

In [ ]:
head_stage_name = "frozen_head"
head_callbacks = build_callbacks(head_stage_name)
head_initial_epoch = get_completed_epochs(head_stage_name)
head_target_epoch = EPOCHS_HEAD

history = None

if head_initial_epoch >= head_target_epoch:
    print(f"Frozen-head stage already complete through epoch: {head_initial_epoch}")
else:
    try:
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            initial_epoch=head_initial_epoch,
            epochs=head_target_epoch,
            callbacks=head_callbacks
        )
    except KeyboardInterrupt:
        print("\nFrozen-head training interrupted. Saving recovery artifacts...")
        save_recovery_snapshot(model, tag="frozen_head_interrupt")
    finally:
        save_recovery_snapshot(model)

head_completed_epochs = get_completed_epochs(head_stage_name)
print(f"Frozen-head stage complete through epoch: {head_completed_epochs}")

Epoch 1/5
789/789 ━━━━━━━━━━━━━━━━━━━━ 598s 713ms/step - accuracy: 0.2248 - loss: 1.6140 - top_2_accuracy: 0.4352 - val_accuracy: 0.2383 - val_loss: 1.6014 - val_top_2_accuracy: 0.5293
Epoch 2/5
789/789 ━━━━━━━━━━━━━━━━━━━━ 502s 636ms/step - accuracy: 0.2347 - loss: 1.6048 - top_2_accuracy: 0.4457 - val_accuracy: 0.1944 - val_loss: 1.6027 - val_top_2_accuracy: 0.4412
Epoch 3/5
789/789 ━━━━━━━━━━━━━━━━━━━━ 507s 642ms/step - accuracy: 0.2333 - loss: 1.6035 - top_2_accuracy: 0.4501 - val_accuracy: 0.2381 - val_loss: 1.6016 - val_top_2_accuracy: 0.4896
Epoch 4/5
789/789 ━━━━━━━━━━━━━━━━━━━━ 0s 603ms/step - accuracy: 0.2389 - loss: 1.6028 - top_2_accuracy: 0.4524

In [ ]:
fine_stage_name = "fine_tune"
fine_completed_only = get_completed_epochs(fine_stage_name)
fine_initial_epoch = head_completed_epochs + fine_completed_only
fine_target_epoch = head_completed_epochs + EPOCHS_FINE
fine_callbacks = build_callbacks(fine_stage_name)

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ]
)

history_fine = None

if fine_initial_epoch >= fine_target_epoch:
    print(f"Fine-tuning stage already complete through epoch: {fine_initial_epoch}")
else:
    try:
        history_fine = model.fit(
            train_generator,
            validation_data=val_generator,
            initial_epoch=fine_initial_epoch,
            epochs=fine_target_epoch,
            callbacks=fine_callbacks
        )
    except KeyboardInterrupt:
        print("\nFine-tuning interrupted. Saving recovery artifacts...")
        save_recovery_snapshot(model, tag="fine_tune_interrupt")
    finally:
        save_recovery_snapshot(model)

fine_completed_epochs = get_completed_epochs(fine_stage_name)
print(f"Fine-tuning stage complete through fine-tune epoch count: {fine_completed_epochs}")

In [ ]:
val_generator.reset()
val_probs = model.predict(val_generator, verbose=1)
val_true = val_generator.classes
unknown_idx = class_names.index("Unknown")

top1 = np.argmax(val_probs, axis=1)
top1_conf = np.max(val_probs, axis=1)
top2_conf = np.partition(val_probs, -2, axis=1)[:, -2]
margins = top1_conf - top2_conf

print("Validation confusion matrix (raw argmax):")
print(confusion_matrix(val_true, top1))
print()
print(classification_report(val_true, top1, target_names=class_names, digits=4))

for threshold in CONFIDENCE_THRESHOLDS:
    final_pred = top1.copy()
    reject_mask = (top1 == unknown_idx) | (top1_conf < threshold) | (margins < MARGIN_THRESHOLD)
    final_pred[reject_mask] = unknown_idx

    accepted_mask = final_pred != unknown_idx
    accepted_total = int(np.sum(accepted_mask))
    accepted_accuracy = float(np.mean(final_pred[accepted_mask] == val_true[accepted_mask])) if accepted_total else 0.0
    acceptance_rate = float(np.mean(accepted_mask))

    true_unknown_mask = val_true == unknown_idx
    unknown_recall = float(np.mean(final_pred[true_unknown_mask] == unknown_idx))
    false_accept_rate = float(np.mean(final_pred[true_unknown_mask] != unknown_idx))

    print(
        f"threshold={threshold:.2f} | accepted_accuracy={accepted_accuracy:.4f} | "
        f"acceptance_rate={acceptance_rate:.4f} | unknown_recall={unknown_recall:.4f} | "
        f"unknown_false_accept_rate={false_accept_rate:.4f}"
    )


In [ ]:
save_recovery_snapshot(model, tag="manual_final_save")
print(f"Best model available at: {BEST_MODEL_PATH}")
print(f"Epoch checkpoints directory: {CHECKPOINT_DIR}")